# 🧠 ระบบกริดของ YOLO และการมอบหมายเซลล์หลายระดับขนาด (YOLO Grid System and Multi-Scale Cell Assignment)

ยินดีต้อนรับสู่สมุดบันทึกคำอธิบายเชิงปฏิบัติสำหรับ **ระบบกริดของ YOLO**! ในสมุดบันทึกนี้ เราจะ:
1. เรียนรู้วิธีที่ YOLO แบ่งรูปภาพอินพุตออกเป็นกริดความละเอียดต่าง ๆ เพื่อตรวจจับวัตถุ
2. สร้างอัลกอริทึมมอบหมายเซลล์ (cell assignment) เพื่อกำหนดว่าระดับขนาด (scale) และเซลล์กริดใดมีหน้าที่ในการตรวจจับวัตถุที่กำหนด
3. จำลองระดับขนาดการตรวจจับสามระดับ: ขนาดการก้าว 8 ($80\times80$), ขนาดการก้าว 16 ($40\times40$), และขนาดการก้าว 32 ($20\times20$)
4. มอบหมายวัตถุที่กำหนดเอง (เช่น วาล์ว และหัวบ่อ) ไปยังเซลล์ที่ถูกต้องโดยอิงตามจุดศูนย์กลางและพื้นที่ของวัตถุ
5. พลอตแผนผังกริดและการแมปวัตถุโดยใช้ Matplotlib เพื่อแสดงภาพแนวคิดการมอบหมายเป้าหมายแบบหลายระดับขนาดให้เห็นภาพชัดเจน

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

## 1. ตรรกะการมอบหมายเซลล์กริด (Grid Cell Assignment Logic)

ฟังก์ชันนี้จะรับพิกัดกล่องขอบเขตและมอบหมายวัตถุนั้นไปยังเซลล์กริดในระดับขนาดเฉพาะ

In [ ]:
def assign_grid_cell(bbox, image_size=(640, 640)):
    x1, y1, x2, y2 = bbox
    w = x2 - x1
    h = y2 - y1
    area = w * h
    
    cx = (x1 + x2) / 2.0
    cy = (y1 + y2) / 2.0
    
    # Stride selection based on area
    if area < 64 ** 2:
        stride = 8
    elif area < 192 ** 2:
        stride = 16
    else:
        stride = 32
        
    cell_col = int(cx // stride)
    cell_row = int(cy // stride)
    grid_w = image_size[0] // stride
    grid_h = image_size[1] // stride
    
    return {
        "stride": stride,
        "grid_shape": (grid_w, grid_h),
        "cell_index": (cell_col, cell_row),
        "center_pixel": (cx, cy)
    }

## 2. การตั้งค่าวัตถุทดสอบ (Setting Up Test Objects)

เรากำหนดค่าส่วนประกอบสามรายการจากชุดข้อมูล PTT ที่มีขนาดต่างกัน และแมปพวกมันไปยังกริดที่สอดคล้องกัน

In [ ]:
test_objects = {
    "small-valve": [100.0, 100.0, 140.0, 130.0],  # Small
    "control-valve": [200.0, 200.0, 310.0, 320.0], # Medium
    "wellhead": [150.0, 150.0, 480.0, 520.0]       # Large
}

for name, bbox in test_objects.items():
    res = assign_grid_cell(bbox)
    print(f"{name.upper()}:\n  Assigned to stride {res['stride']} grid {res['grid_shape']} cell {res['cell_index']} at center {res['center_pixel']}\n")

## 3. การแสดงผลการแมปกริดด้วยภาพ (Visualizing Grid Mappings)

เราจะพลอตวัตถุแต่ละชิ้นเทียบกับชั้นกริดที่ได้รับมอบหมาย โดยจุดสีแดงแสดงจุดศูนย์กลางของวัตถุ และกล่องสีน้ำเงินที่เน้นสี (หรือสีแดงที่แรเงาในภาพย่อย) แสดงถึงเซลล์กริดที่รับผิดชอบการทำนายวัตถุนั้น

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
names = ["small-valve", "control-valve", "wellhead"]
colors = ["green", "blue", "purple"]

for idx, ax in enumerate(axes):
    name = names[idx]
    bbox = test_objects[name]
    res = assign_grid_cell(bbox)
    stride = res["stride"]
    cell_col, cell_row = res["cell_index"]
    cx, cy = res["center_pixel"]
    
    ax.set_xlim(0, 640)
    ax.set_ylim(640, 0)
    
    # Draw object bbox
    w = bbox[2] - bbox[0]
    h = bbox[3] - bbox[1]
    rect = patches.Rectangle((bbox[0], bbox[1]), w, h, linewidth=2, edgecolor=colors[idx], facecolor='none', label=name)
    ax.add_patch(rect)
    
    # Draw responsible cell grid boundary
    cell_x1 = cell_col * stride
    cell_y1 = cell_row * stride
    rect_cell = patches.Rectangle((cell_x1, cell_y1), stride, stride, linewidth=2, edgecolor='red', facecolor='red', alpha=0.3, label='Responsible Grid Cell')
    ax.add_patch(rect_cell)
    
    # Plot center point
    ax.plot(cx, cy, 'ro', markersize=8, label='Object Center')
    
    # Add subset of grid lines for visualization
    for val in range(0, 640, stride * 2):
        ax.axhline(val, color='gray', linestyle=':', alpha=0.3)
        ax.axvline(val, color='gray', linestyle=':', alpha=0.3)
        
    ax.set_title(f"{name} on Stride {stride} Grid", fontsize=12)
    ax.legend()
    ax.grid(False)

plt.tight_layout()
plt.show()

## 4. ประเด็นสำคัญ (Key Takeaways)

-   **การมอบหมายเซลล์เดียว (Single Cell Assignment):** แม้ว่าวัตถุจะมีขนาดใหญ่และครอบคลุมหลายเซลล์ แต่จะมีเพียงเซลล์ **เดียว** เท่านั้น (เซลล์ที่มีจุดศูนย์กลางของวัตถุอยู่ข้างใน) ที่มีหน้าที่ทำนายวัตถุชิ้นนั้น ซึ่งจะช่วยหลีกเลี่ยงการสร้างเป้าหมายที่ซ้ำซ้อนในการฝึกฝน
-   **การรวมระดับขนาด (Scale Aggregation):** การมีกริดหลายความละเอียดช่วยให้ YOLO สามารถเพิ่มประสิทธิภาพของแผนที่ลักษณะเด่นได้: กริดขนาดใหญ่ (stride 8) จะเก็บรักษารายละเอียดเชิงพื้นที่ในท้องถิ่น ในขณะที่กริดขนาดเล็ก (stride 32) จะเก็บรักษาข้อมูลบริบทในภาพรวม